In [ ]:
# Give the operations app's service principal access to the lakefed_ingest Lakebase database.
# Run as the instance manager and tables' owner (whoever runs lakefed_ingest_setup). Idempotent:
#   1. CAN_USE on the Lakebase instance (so the app can mint its own short-lived credential);
#   2. a Postgres role for the service principal (created if missing);
#   3. table and sequence privileges in schema lakefed_ingest, plus default privileges for tables the
#      same owner creates later.
# (This replaces an app `database` resource, which some workspaces refuse to attach.)
#
#   4. USE CONNECTION on each connection in source_connections, as whoever runs this job (a connection it
#      can't grant on is reported; the rest are granted).
#
# Privileges: SELECT, INSERT, UPDATE on all tables (the app reads telemetry, edits control rows and
# flips checkpoints to reseed) and USAGE on sequences (inserting control rows). No DELETE.
dbutils.widgets.text("app_name", "")
dbutils.widgets.text("db_instance", "lakefed-ingest")
dbutils.widgets.text("db_name", "lakefed_ingest")
# Unity Catalog connections the app discovers through (comma-separated): it is given USE CONNECTION on each.
dbutils.widgets.text("source_connections", "")
app_name = dbutils.widgets.get("app_name")
db_instance = dbutils.widgets.get("db_instance")
db_name = dbutils.widgets.get("db_name")
source_connections = dbutils.widgets.get("source_connections")
assert app_name, "app_name is required"

In [ ]:
import uuid
import psycopg
from psycopg import sql
from databricks.sdk import WorkspaceClient

from databricks.sdk.errors import NotFound
from databricks.sdk.service import database as dbs
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

w = WorkspaceClient()
role = w.apps.get(name=app_name).service_principal_client_id
assert role, f"app {app_name} has no service principal yet; deploy the app first"

# 1. instance access (CAN_USE) for the app's service principal
w.permissions.update("database-instances", db_instance, access_control_list=[
    AccessControlRequest(service_principal_name=role, permission_level=PermissionLevel.CAN_USE)])
# 2. its Postgres role
try:
    w.database.get_database_instance_role(instance_name=db_instance, name=role)
except NotFound:
    w.database.create_database_instance_role(
        instance_name=db_instance,
        database_instance_role=dbs.DatabaseInstanceRole(
            name=role, identity_type=dbs.DatabaseInstanceRoleIdentityType.SERVICE_PRINCIPAL))
host = w.database.get_database_instance(name=db_instance).read_write_dns
cred = w.database.generate_database_credential(request_id=str(uuid.uuid4()), instance_names=[db_instance])
conn = psycopg.connect(host=host, port=5432, dbname=db_name, user=w.current_user.me().user_name,
                       password=cred.token, sslmode="require", autocommit=True)
cur = conn.cursor()
cur.execute("select 1 from pg_roles where rolname = %s", (role,))
assert cur.fetchone(), f"Postgres role {role} was not created on {db_instance}"
r = sql.Identifier(role)
statements = [
    sql.SQL("grant usage on schema lakefed_ingest to {}").format(r),
    sql.SQL("grant select, insert, update on all tables in schema lakefed_ingest to {}").format(r),
    sql.SQL("grant usage, select on all sequences in schema lakefed_ingest to {}").format(r),
    sql.SQL("alter default privileges in schema lakefed_ingest grant select, insert, update on tables to {}").format(r),
    sql.SQL("alter default privileges in schema lakefed_ingest grant usage, select on sequences to {}").format(r),
]
for s in statements:
    cur.execute(s)
conn.close()
print(f"granted app {app_name} (role {role}) access to {db_instance}/{db_name}.lakefed_ingest")
# 4. USE CONNECTION on the connections it discovers through (connection_grants.py, in the app's src folder)
not_granted = {}
if source_connections.strip():   # none named: the job does exactly what it did before
    import os, sys
    nb_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    sys.path.append("/Workspace" + os.path.dirname(nb_dir) + "/src")
    import connection_grants
    for name, error in connection_grants.grant_use_connection(w, role, connection_grants.connection_names(source_connections)).items():
        print(f"USE CONNECTION on {name}: " + ("granted" if error is None else f"NOT granted: {error}"))
        if error:
            not_granted[name] = error
if not_granted:
    print(f"{len(not_granted)} connection(s) not granted: their owner grants USE CONNECTION to {role} in Catalog Explorer")
dbutils.notebook.exit(role)